# Stardox Email Scraper

Takes a list of GitHub usernames and scrapes their public email addresses using a headless browser.

**How it works:**
1. Spins up headless Chromium via Playwright (downloads its own browser — no system Chrome needed)
2. Visits each user's GitHub profile
3. Looks for email in the profile sidebar (JS-rendered)
4. If no email on profile, checks their commit history (.patch files)
5. Outputs username:email pairs as a downloadable CSV

In [13]:
# Install Playwright + its own bundled Chromium (does NOT use system Chrome)
!pip install -q playwright nest_asyncio pandas tqdm
!playwright install chromium
!playwright install-deps chromium

Installing dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is already the newest version (20120503-10build1

In [14]:
import re
import asyncio
import nest_asyncio
import pandas as pd
from tqdm.notebook import tqdm
from playwright.async_api import async_playwright

nest_asyncio.apply()

EMAIL_RE = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')
IGNORE_PATTERNS = ['noreply', 'users.noreply.github.com', 'github.com', 'githubusercontent']


def is_valid_email(email):
    if not email:
        return False
    email_lower = email.lower()
    for pattern in IGNORE_PATTERNS:
        if pattern in email_lower:
            return False
    return True


async def start_browser(github_cookie=None):
    """Launch headless Chromium via Playwright async API."""
    pw = await async_playwright().start()
    browser = await pw.chromium.launch(headless=True)

    if github_cookie:
        context = await browser.new_context()
        await context.add_cookies([{
            'name': 'user_session',
            'value': github_cookie,
            'domain': '.github.com',
            'path': '/',
            'secure': True,
        }])
        page = await context.new_page()
        print('Browser started with GitHub session!')
    else:
        page = await browser.new_page()
        print('Browser started (anonymous — profile emails will be hidden)')

    return pw, browser, page


async def stop_browser(pw, browser):
    """Clean up browser and playwright."""
    try:
        await browser.close()
    except Exception:
        pass
    try:
        await pw.stop()
    except Exception:
        pass


async def scrape_email_from_profile(page, username):
    """Visit GitHub profile and extract email from page text."""
    try:
        await page.goto(f'https://github.com/{username}', wait_until='networkidle', timeout=20000)
        await page.wait_for_timeout(2000)

        body_text = await page.inner_text('body')

        emails = EMAIL_RE.findall(body_text)
        for email in emails:
            if is_valid_email(email):
                return email

        source = await page.content()
        mailto_matches = re.findall(r'mailto:([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})', source)
        for email in mailto_matches:
            if is_valid_email(email):
                return email

    except Exception:
        pass

    return None


async def scrape_email_from_commits(page, username):
    """Get email from user's commit .patch files."""
    try:
        await page.goto(f'https://github.com/{username}?tab=repositories&type=source',
                         wait_until='networkidle', timeout=20000)
        await page.wait_for_timeout(1000)

        repo_elements = await page.query_selector_all('a[itemprop="name codeRepository"]')
        repo_names = []
        for el in repo_elements[:3]:
            name = await el.inner_text()
            repo_names.append(name.strip())

        if not repo_names:
            return None

        for repo_name in repo_names:
            try:
                await page.goto(
                    f'https://github.com/{username}/{repo_name}/commits?author={username}',
                    wait_until='networkidle', timeout=20000)
                await page.wait_for_timeout(1000)

                commit_links = await page.query_selector_all(
                    f'a[href*="/{username}/{repo_name}/commit/"]')

                for commit_link in commit_links[:5]:
                    href = await commit_link.get_attribute('href')
                    if not href or '/commit/' not in href:
                        continue

                    label = (await commit_link.get_attribute('aria-label')) or ''
                    text = (await commit_link.inner_text()) or ''
                    if 'merge' in label.lower() or 'merge' in text.lower():
                        continue

                    if href.startswith('/'):
                        href = 'https://github.com' + href

                    await page.goto(href + '.patch', timeout=15000)
                    await page.wait_for_timeout(1000)

                    page_text = await page.content()

                    from_match = re.search(r'From:.*?<([^>]+@[^>]+)>', page_text)
                    if from_match:
                        email = from_match.group(1)
                        if is_valid_email(email):
                            return email

                    emails = EMAIL_RE.findall(page_text[:3000])
                    for email in emails:
                        if is_valid_email(email):
                            return email

            except Exception:
                continue

    except Exception:
        pass

    return None


async def scrape_email(page, username):
    """Try profile first, then commits."""
    email = await scrape_email_from_profile(page, username)
    if email:
        return email
    return await scrape_email_from_commits(page, username)


print('Functions loaded. Ready to scrape.')

Functions loaded. Ready to scrape.


In [15]:
# ===========================================
# GITHUB SESSION COOKIE (required to see profile emails)
#
# How to get it:
# 1. Log into github.com in your browser
# 2. Open DevTools (F12) -> Application -> Cookies -> github.com
# 3. Find the "user_session" cookie and copy its value
# ===========================================

GITHUB_COOKIE = ""  # paste your user_session cookie value here

# ===========================================
# PASTE YOUR USERNAMES BELOW (one per line)
# ===========================================

usernames_input = """
3rdAI-admin
ArkayaVenture
Kaairofelipe
cpiprint
cpiprint
cpiprint
cpiprint
shreyasgm
asfakahamedc
jsairdrop1
Kakachia777
zfogg
BinxAI
aiinpocket
GiantClam
lordwilsonDev
Mirxa27
jrmatherly
anjijava16
amp135791
danmestas
bobdavis84
Sealjay
TeamADAPT
David2024patton
yubarrdevo
idgrou
mundo-labs
brandonlacoste9-tech
xiaomihu5460
Hyperi0nGit
jrmatherly
cogpy
cogpy
wandb
Glitterstrafe
bobdavis84
bobdavis84
LelandParker
ChatAndBuild
grandeurmedia
PlawIO
secureonelabs
kodustech
taskcrew
aknibircse
aknibircse
BuDozKeN
dustland
AlaqmarG
shreyasgm
tomcounsell
kushalBanda
WomB0ComB0
acrinym
9cog
GlacierEQ
GlacierEQ
GlacierEQ
GlacierEQ
theexperiencecompany
vanniekerkcf
browser-use
stkzlv
tajmahal226
ohana-garden
sheeki03
groupthinking
robertpelloni
alpharomercoma
BY-SOMMER
yuichiinumaru
brandonlacoste9-tech
9cog
hurdcog
9cog
hurdcog
9cog
9cog
9cog
scszcoder
CrazyDubya
trizist
yonasTMC
brandonlacoste9-tech
ayeedd
EpicStaff
POWERFULMOVES
POWERFULMOVES
POWERFULMOVES
doehyunbaek
Laihiujin
ccwu0918
ccwu0918
sizzlebop
sizzlebop
sizzlebop
vijethph
xiehuangzhijia
rabbykst
Shajan
nodetool-ai
Jeff-Kazzee
jubenitogarcia
Anujpatel04
bilbywilby
howaiconnects
Yuvraj-Dhepe
orgitcog
melbazpeach-source
li-boxuan
andyjm2k
MarkdownMind
plotsndots
didierhk
NizamSaidin
ThePhoenixAgency
AaronJessen
AaronJessen
SocioProphet
TheSamurai1
SocioProphet
MADdegen
agisotadev
TeamADAPT
Vikaash301
ssushant0011
ssushant0011
ssushant0011
ssushant0011
Josepavese
Zunnesco1
usagiuzumaki
markl-a
executiveusa
kalchakra13
kalchakra13
o9nn
matheusmaldaner
oldmangrizzz
omerakben
FigmaAI
dre8597
tanbaycu
kritsanan1
fleet-ai
Yasir-Hasnain
4xiaxia
JSTONE1111
cbwinslow
grayswansecurity
credli-X
terminal-agent
ErwanLegrand
cliztech
cliztech
catoailabs
deaspo
o9nn
o9nn
aws-samples
alexmcnee84
lucas-explica
o9nn
sheevu
mrsakrano
yangguo
o9nn
AIzentek
DOC-Painting
sgInnora
AI-Development-Studio
CatalinCighi
appdevwk
appdevwk
agent-infra
team-telnyx
Jineshkumar1
abdulrahman305
andreathar
aws-samples
cogpy
Fastiraz
jeunjetta
AI-Development-Studio
Cloud-Curio
LunaeAI
cogpy
cogpy
cogpy
cogpy
gfrakkinbaltar
gfrakkinbaltar
csrinu15
csrinu15
xxosproject
robertpelloni
Marvis-Lab
ParentSquare
kalil0321
POWERFULMOVES
bb-boy680
cogpy
mcxlab
bozza-man
ast-pub-fiber-flask
AIComputerServices
HarleyCoops
Terresapan
ast-pub-fiber-flask
BizraInfo
BizraInfo
BizraInfo
suissa
hoanganh-hue
hoanganh-hue
hoanganh-hue
enyst
twonfi
lordamos
cogpy
cogpy
cogpy
damirbarr
Nexclair
nottelabs
opea-project
fork-the-planet
dazeb
OpenHands
NicoAipro
Nico-Ai-corp
sharmabinaya
OpenHands
Bakk21
Zalos-io
autoppia
Rambo1985
DASTOVEK
DASTOVEK
ranjankumarpatel
BjornMelin
org-echo-opencog
abcmartin
OpenCoq
vibhorjoshi
MadScientist85
Metro5000SDx
SaschaHeyer
HandyKnox
HandyKnox
popupempire
nicklatrick
fAi-84
Kemei-Tech
Daelyte
Daelyte
Daelyte
GizzZmo
alishangtian
vikramsg
mattsk123
ANT0071
ANT0071
Cordycepsers
Cordycepsers
ydennisy
RemyLoveLogicAI
cua-framework
JoseLFernandez
JoseLFernandez
nodetool-ai
cdalin1985
instabase
yuanweize
montypylons
mycguo
tariqjamil-bwp
saxster
jgeofil
charithmadhuranga
sendralt
Gauravs-2k
angular
akiraueda87
unify-apps
orby-ai-engineering
JrmyDev
ycechungAI
ycechungAI
zyxcambridge
HarishKanna-05
rafaelbeckert
ZoneCog
ryanalmb
micah-firstbuilder
browser-use
ZUOLI11111111
imlovepeacezen-art
jnystrom14
toolsdk-ai
nullroute-commits
spiralgang
spiralgang
flanzipit
rodolfoiron9
chosen8823
Sherlock999xxx
rustam7177
ArkayaVenture
AReid987
Centaurioun
Arindam200
passariello
RahulSaini02
KowaiAI
green-labs
jlfguthrie
HarxSan
jcowhigjr
TanNhatCMS
SPThole
JacarendaLabs
OscaeGTX
senchi-ins
00hello
aws-samples
Ashrokss
CarlosIvars
zk1tty
rodgui
browser-use
AYUSHKHAIRE
sizzlebop
gonsoomoon-ml
Mosasathaliya
OzCog
outskill-git
aipotheosis-labs
kavin525zhang
TheEpTic
buildfastwithai
OzCog
AReid987
MDalamin5
ishaheen10
vapordude
awslabs
Oliver-Yang-1
dipjyotimetia
mawwalker
yiju-zhao
OpenCoq
The-Swarm-Corporation
jamsturg
EvoAgentX
TrainLoop
365cent
wwwediai
DenisAIagent
OzCog
inclusionAI
dsdtsolutionsllc
dsdtsolutionsllc
jadenblack
ZoneCog
dsdtsolutionsllc
mcp-research
johnson-52197
snehinsen
darkangelpraha
OsamabinAdnan
HyperCogWizard
thebesteric
pixillab
pixillab
arnavsurve
Jcheng777
Addaitya
netguycode
cumthyb
s3cr1z
lazycoder1
alibaba
Emadalshamery
balaganesh102004
sauravpanda
Martyparty1988
CheukYuen
WWI2196
ayushshivaji
hestami-ai
PrimeDeviation
AGI-FBHC
sauravpanda
sauravpanda
earth-sol
praneethravuri
WildlifeDEE29
kunwarVivek
iammultiman
Tehreem-Asghar
Raghu6798
davideconsonni
yeyos300love
admin-invitedekho
will7455
browserbase
Mozilla-Ocho
invisible-tech
DannyRuchtie
convergence-ai
secureonelabs
Sonatrix
mauriciochaiben
GrimFandango42
kaunda-a
maxmax1992
akdeepankar
arghya05
sharrifhussain
balaganesh102004
AKSarav
balaganesh102004
LestreZhao
manglesh-nimbalkar-apptware
davidrsetti
twang849
KrunalShindeMF
browser-use
tronschell
zhanfang
iyangming
sahilanand2412
relentless123
NishantGarg13352
Jake-Nguyen94
ethanzhrepo
openrijal
mrarejimmyz
jink-ucla
idkbruhjbvskbg
KhuKhusokha
Nawsh1337
longmazhanfeng
paulpham157
paulpham157
AVISEBBAH2023
pravinkannan18
isavchenkoivan
godspeed-003
hexdocom
Pcnaid-Dev
aaronnat23
DAGS-data
Mohamed-amine-bouguerra-data-eng
Aadilmalik70
MarneniRoopesh
max-sydorov
ozzi0221
moncifem
project-zelash
jsgomezs1
hackclub
uspraveen
adityamwagh
oda251
sapoepsilon
Shallum99
dicacid
elisaribeiro
boshjerns
esc-ai-dev
yogesh127
Darwin-Chow
boshjerns
PerVillalva
yumpyy
Saichandu1845
kitadmin01
danteindrex
caicongyang
xiaozeng2296
Jaystrikesback
yelyshch
snobu
GauravJiandaniGJ
Computer-use-agents
Shreya20002
sendralt
yincangshiwei
galvin59
Guido1Alessandro1Trevisan
anupmanekar
Paul-Appleby
AbdooMohamedd
Fr0ndeur
voxmenthe
BethanyJep
deciduus
Arinraja
rshdeka
quickhdsdc
korwabs
yongjun-0903
livelybug
LucasGu-Star
blkout-hd
srmarkie
anupmanekar
abaveja313
Ksotillo
BharathLakkoju
ik2535
Dimvy-Clothing-brand
RadyUX
beautifulboy9527
tridungduong16
q33566
eternalai-org
NagyVikt
faizsameerahmed96
gcohen1928
longwarriors
kinqbert
fishke22
jcompanion
iamulya
albertnahas
xyuzh
browser-use
Aadilkhan-og
Vivek-120604
veerdoshi
00000DaveX00000
AA-Turner
Gayathri-Thummuru
benzdriver
leowom
orpaynter
CesarMaia2025
bizob2828
amaze1111
lijiaxing1997
DGSI-UPC
hg3386628
hibye-ys
anuragsingh132200
m-germano
yomariano
shirazkk
kiante-fernandez
merdandt
bmoffitt1990
Guido1Alessandro1Trevisan
mattosmeerkat
voxmenthe
ManojINaik
jomarcello
corysimilarweb
dnivra26
Eshikamahajan
nibura2002
DouglasXiao
kevinflynn0503
RomitDeokar
Shunya-OMORI
rikkooo
Sujal-S
gauravmodak2001
paras55
capbility
RiccardoRR57
fuyukitn
phonhay103
AyushSid28
peckermann12
nishimu555
archana014
AboutSBC
dloperab
LucaPaganin
Adharsh-goud
wanghuaibao
mianzhehuang
Nikith5949
lezhdz98
saikumar-everest
shubcodes
damianvtran
humble92
rubaahmedkhan
ramziibrahim
JckHoe
leonborsato
Alielka0x404
anjali1-boddu
marshallvoid
iamraiky
morellovich
shashankboosi
LHKong7
mikey-wang
SARAMALI15792
a-code-a
mnkillebr
ajayatwal1105
albertnahas
elmerson15
mengbo
AliNikoo73
Amrit1810
kielni
Amit2yzx
silanthro
josdoaitran
dylangroos
oluwaseyiTaiwo
A-P-Shukla
Rohit0812
Mirxa27
Jiarui1111
jasmithaparasa17
pietromarini00
mindblowngaming
mindblowngaming
yordanoswuletaw
spexxxzzz
robinmanuelthiel
MustafaAgentBuilder
SuperAce100
JVonBorstel
muhammadsami987123
khurrameycon
TSheylock
VedanshiShah7
photo-yuji
mac999
tensimixt
Rishabhsingh78
sprakhar778
Pravallika283
arun477
mcp-research
vishalpal1997
MatthewLaw1
nodetool-ai
prakersh
alhendy56
Gowri173
bingal
Qiue-G
ZTE-AICloud
stanford-mast
jmarusak
zihaomo080808
AndreFGard
Cristiano-Rocha
alesanchezr
abhinav262666
Dhilan-Panjabi
kc099
4nur4g
EchoCog
EchoCog
Loadsure
zhangjian-ai
Peleton-011
scullzz
himanshu188
weiting-tw
ahippoly
jeffliulab
yash25112003
mcp-research
JackLGB
mohann1234
cometa-rocks
effective-pl
zhangzic1
mahyancheng
Edusprangoski
umuo
anupmanekar
mcp-research
mcp-research
mpoff3
ztobs
Chen-zexi
ekolivero
183600
Mahmoud-A-Noor
rikijha
Harsh4949
officialchengyud
Rudegrudes
ishanyash
MIsfitMatty
ihongs
anders-hopland
saravatpt
anishgoswamiEXL
Adityajeet
askzash
chanrute
easy-easy
kamendula
Aravindkumar-Rajendran
blackandragon
brainmatt
martinhabaj
abhinav-raj-dev
Mathieu-Hexamind
KoenScript
ANeuronI
howya
wangjiangbo-00
pseudo-longinus
carlosmb2023
jon-chun
chikamsoachu1
chikamsoachu1
Jeezlouis
jeongsk
derekmeegan
bravelovezx
sayliakarshe
mahyancheng
malhotratanuj
Rahulg321
riagusmita182
vishalkoc2016
tomiyasu0428
Abdullahiqbal04
tkc310
mahyancheng
akihiro-sakamoto-safie
MoriEdan
ENdonga
StefanDevstar
Rajesh9998
edulechuga
ahadnaeem785
sarperarikan
praveenhm
treoa
patrickhulce
2025NKUCS-agent
RenanKohler
Rajesh9998
LeapLabTHU
pietrozullo
ekinohito
dannyread
liguopings
Jbae04
mantrakp04
kym6464
aboutvlads
arnavvaryani
sujitojha1
aish-am22
cydonianbanana
duongthuy125
carlosplanchon
abdibrokhim
akhatua2
DheemanKumar
xoity
webagent-cloud
femto
naitikmalaviya
Kalyani0131
CAI991108
JessicaChildress
shriraj-m
LEVI-DEVIA
silanthro
masrialx
VanTXDev
veedaisme
shadanxd
chin-jlyc
Manideep-Kanna
jeffliulab
Shubhamsaboo
garylab
Saad-Mufti
satishk01
S5432
sh-riyad
rahulnovarroh
marziasu
GooglePhone
G-Narendra
licy-123
Beehive324
philmui
urimem
zloeber
chunchiehdev
Ovie-Eharisi-Ayomah
EngineerProjects
BanDianMan
od41
beijijizhou
timonharz
edonyzpc
Aicode42
Venkat5674
Aryan-210780
jthickma
Aryan-210780
Vadavision
Boo712
TheOzymandiasHimself
nottelabs
espin086
Aryan-210780
datascienceManager
zestor
ivecoGolden
rfahrn
marshallhamelton42
sanjion
YeonwooSung
cjhargreaves
hypo69
bilgoun
Tongs2000
sailorjs0804
sinzy0925
AshwinPushpad
Sachin-Bharadwaj
ahmedsheraz2025
Sher110106
jackljk
rakesh-patel-nzl
ArifAhmed120829
rorygeddes
eight-atulya
Juliosimoes199
f6844710
khanasif1
simrat12
mnkillebr
k4nkan
yamamoto-ayano
lugi0
Marvin-MM
MinimalFuture
bondzhan
DiengWinz55
krkavinraj
enkhbold470
adricwht
RafaelZelak
KanalaJayareddy
dadsec-dev
aliashkov
sriram-everest
myhendry
LuisPizarro04
chethanuk
jakerqin
xobust
krishna22112023
Vaishnavi024
falconlee236
shathwik94
micaelleos
battucave
awkwardindustries
NabilAziz99
rambawankule
cuongducle
JinYSun
1137285095
Go-XiaoPeng
NishilG
harsh16629
MehdiAliouan
CoderTag
yianan261
Startup-Consulting
Enigma10
rytrix
JeffersonRiobueno
JohannaZheng
praveenmaragani22
hoyirul
WeatherPal-AI
ascotai
ruska-ai
Rossember555
mrnunfungible
AIchovy
ccozad
adrieljoshua
Anish-Reddy-91
irl-os
ArtiLife-Dev
RemyLoveLogicAI
RemyLoveLogicAI
kuttysoftmy
kuttysoftmy
phulelouch
vikas434
Surya-R-1999
uypapi
chiseanchang0727
mintedmaterial
sudofixit
POC-2025
KishanKokal
SreeSharvesh
nickmitreski
sgsen
rpm-vectorial
JitendraJSM
Rajanavee
Krish-Goyani
Nikithayadav25
UmarHayat15
InCoB
dhakarRaghu
Josueperez0225
banu-teja
tangzi
franskey-0112
winter1203
salerscub
S5432
NextDoorLaoHuang-HF
RongjieChen
ew384
LouisCan
enterme2
ezypzylemon
borgius
mnkillebr
zhibuyu
SatyamSingh8306
Di-Is
dhruv1710
mjm3853
AdiGaikwadNewpage
phuctvu
Harika-dungala
jawadefaj
Ren-0227
hireshBrem
refreshdotdev
Joselier3
Octavious
eren23
dchanda50q0on
heyibad
basiches
eepson123tw
alexsirait
asvskartheek
IsraelBezerra48
riccardo-larosa
JoaoCarabetta
bsantanna
CaptainIgl00
igor-pysmennyi-kpi
BhanuTejaKammari
mihir-kanzariya
amitbhalla
Pavankumar599
alishangtian
DoggyCabbage
crusader-initial
postbird
dbaoir2024
adcwa
Enetact
tunjiNg01
Solsolol
harshal-96
weifengHuang
entrepeneur4lyf
nez02
0N3X-Tech
Lumen951
drzo
MattiasKDev
illegalDragon47
summerpunch
carolzy
adamjen
Cdatainsights
JouBarzdukas
conorbranagan
Allmight-456
karlcarlo
imann24
SDU-Gary
boxabirds
tj-scripts
selfepc
fedollo
ssillerom
wilsonaustin10
SamraAzizi
TianMingXTU
newworld123-max
devleks
NanoAgentCode
jokyun
Miaque
24704104
Yat3s
lancejames221b
chebon254
lleoMarquees
vetlefo
vetlefo
welli7ngton
HiteshS08
isaccanedo
Adolmeal
Guisong-Fu
boraaaydin
Machine-Learning-ML
johnymephisto
hackerinheels
debsouryadatta
AniketKakde04
orpaynter
Tslilon
LoveyJyhyun
block-88
prompted365
Aircraft-carrier
Puggo1145
tkubica12
505labs
11cafe
akshay-maryala
alleneee
sharu28
thanhlamliam
henry0hai
route250
hwillGIT
mihai-pompiliu
ThamuMnyulwa
darshan-naidu21
devilanirudh
Sanjana-Gajula
kenotron
svceadboys
vishal-parekh
sumitrathore1313
rockmuhil
Please-just-dont
666keke
FelipeAriasT
chandrahasM
ltejedor
chupa-cabras
Danyw24
vaishnavidesai09
shin5ok
methunraj
hellocym
mendo9
Praveenpadidapu
aravindinduri
AdrianLuk12
jakerobers
Damandeep1313
AloyBanerjee
gazitanbhir
Barmantanmoy2211
jayezhou
kazuya2400364
tap666
dkchauhan
davydany
sturgis-steele
shivashutosh
franztao
BurnyCoder
arthrod
arthrod
phppho
mhm22332
elongl
StarlightSearch
n0d3m4g0s
kaurson
TongTong313
snowage
tusshaarpd
dongsuo
saintzema
pcliu
TimHub88
Saikiran450
nesthivep
Vedhasb
ZundamonnoVRChatkaisetu
netbity
muhammadzubair112
siliconuy
stevengonsalvez
m-sec-org
arkokoley
mzhl1111
kliewerdaniel
Genjuzu
hcucu163
miketropi
Tehen1
ateebahmad20
xoity
nitesh-chauhan-b
fhormot
shridayal
nishanthswaroop304
walpartei
Pranayvarshan
Sunwood-ai-labs
DevRickLin
karim0sec
microsoft
umiyuri777
lglove
arreyanhamid
ashley-ha
alt-research
natarajan0007
mikegehard
Dwonczykj
hugobergfan
BVISHNU78
BassemMonla
ltejedor
haru-works
sigmabotech
Ketan-K17
snowage
flosrv
praveensharmajava
CarlOwOs
Jeongbyungkyu
ageborn-dev
AminePro7
enablerdao
Dicklesworthstone
ShayaanMalik26
jaeyunha
brunofuentes
iamwille-dev
vinayak-mehta
RuslanTsykaliak
sindhukomsani
hrishabhayush
usdvvv
Ahmad-988
Hanshika11
CarlOwOs
RD945
ArthurBrioche
vishalh29
RichardGuochao
adamwdraper
circulus-tech
hugobergfan
Kirill456Z
Vistaminc
cr7258
sujansaitej
mmrech
clothd
ababdotai
BurnyCoder
IQLynxAI
bansalsahab
the-syn-ai
akshayagaje
Dasari-Pranay
Jayashree1743
Sindhujachikati
michaelodafe
Hajime-Y
AbhishekSharma-17
Oli2861
Fearlessatma
tohid4n
harsh-march123
kngzzz
imshabin
sabbir-wp
zhuyingqin
pelluru
femto
vigiani1
RohitNagareddy
nanxstats
Jeongbyungkyu
mertdemir0
banu-teja
10xshivam
mcseali
OldArchieve
kjliao
bigmoletos
wincap
PrimeDeviation
PrimeDeviation
shashadehuajiang
CarlosMabrey
harsh-march123
Yiyuan-24
zapabob
rafmarimon
ChiragGoyal07
swapniltamse
pranav7
Pablo305
SRIHARISH03
PatrickAbainza
mofa-org
matheus896
Dbbc00
abhishekkadavil
offsideAI
algonacci
sajithamma
Avis2912
Venkatateja11
maaghaa
Jeongbyungkyu
battucave
imaddde867
jlalmes
geeknoobie
mehradans92
aradan207
ShisuiCode23
gmh5225
str02
coolboylcy
TESD-Tech
m1koj
coolboylcy
SarthakML205
larrykoo711
homermeng
shruthitamaraana
mrxvision97
abhipi
ToMatrgod
xuancanhit99
iamkrishnagupta10
petermiller310
vspaswin
Osly-AI
NikhilGiri29
Akhila-1703
xiaozhch5
jim110120
Nobukins
middleamericahomes
PIN-AI
Mark850409
Shybert-AI
middleamericahomes
kalyani324-allu
harshithachilamakuri
xinzhel
Drwaish
lizuyi-6
huqianghui
drzo
echogao2023
cantbeshaky
The-Code-Labz
huodesheng
uptopmeme
jacesca
189569400
middleamericahomes
chuckyLeeVIII
ReeceHarding
Akhila-1703
JanumpallyAkhila
mskbysh
bpawnzZ
kontext-dev
ccvvx1
jinghunsanzu
mixiazhiyang
DroxDynamics
zaldivarmena
poorna075
FoundationAgents
haru-works
codinglearner24
sakshi1499
Ludvig-Hedin
chieftn
saltsami
Sheshadri06
Boddu-Durga-prasad
deepakanroman
whyashthakker
jonnyhoff
bilaltahseen
cat1sobe
maccam912
ChristopheHvd
Studyhive2027
kght6123
JuelHossain
Project1Dev
Ludvig-Hedin
imars
Purushotham-K
debdutta-chatterjee
KelvinQiu802
Akhil18git
jagrat7
CornKT
riteshsukhi
vishal26-rai
falkosch
umshere
lolrazh
devapraveenk
patched-codes
sradc
Hassan-Mehmood
adrianblade
Karthi-1211
vinothg2309
n8thantran
Andry-Arthur
nimitmk7
Profilist
Subhanamir19
ericzakariasson
wilsonaustin10
DamianCapdevila
aaqibali1
WaiGenie
houchimu
ParthGandhi
madmax-10
Yaswanth191
sontl
ruban-raja-anbalagan
raccoonaihq
hrishabhayush
derek2403
rax215
FiresJoeng
dheeraj0000
microvn
Studyhive2027
aditya-aim
fening
Maryala-Harshitha58
kry0sc0pic
akhilsarvi
Pavanchaitanya72
NhatAnh1708
m0nq
PavithraMunagala
D3villl
steel-dev
tonnitommi
abubakarkhanlakhwera
sanjeed5
Saranya-max-ux
sanjeed5
PavaniKotipalli29
srivallirachuri
EvanNotFound
Artvr1t0
Thrishal1105
TBAHRITI-MED
alvarosola1
murtihash
murtihash
smwilfon33
dharshan17sn
gnix45
rachidelctro1
vishnuvardhan105222
MUSTAFA892
dernestbank
Cirilcetra
aswathas
KavyaBS123
mr-chaitu07
SAIKUMARVANKALA
imrobintomar
DilipPusarla
jasonwcfan
HackathonChallengeTeam2025
felipemateus174
andyguzmaneth
praveenvarghese
Gaket
sanghvigaurav95
saikiran369369
Ankitkushwaha90
hemasriram111
mubashir-ehsan
dodlasarika
piopio314
harshaterala
shivanipuppala12
Dhilan-Panjabi
Franck5772
mingxuanchen778
Puneeth18080
lightmen19
hal9ai
Shubhwithai
codewithhamzei
dylantarre
dylantarre
h-gunasekara
gbusto
uypapi
HAMzAliKj
HyperbolicLabs
Thejas775
MukeshAofficial
syedrz
chetan251088
chetan251088
grinssamba
josoroma
fhinkel
Jakub3628800
Pauullamm
codewithhamzei
HAMzAliKj
tsubouchi
Anti-Cult-Dev
Anik2901
paquino11
aj47
lakiet1609
Mahir-Isikli
yumuranaoki
JoshuaLelon
Jacobito2001
BharatBheesetti
jaju
jerichoBob
Bollo444
amediantsev
mesubasi
Surajkadamr
falahgs
ritesh1731
mkinf-io
timeline-holdings
SwastideepKhuntia
PragathTSiva
dejonghe
sergioalberto
gokborayilmaz
CogitoNTNU
VianneyMI
son-n-pham
jleonelion
Jeba-Jebarsan
muhammad-bassiouni
mkinf-io
gokborayilmaz
john-mwangi
Zainali5
duvillierA
oponcefranco
mohankumargupta
gokborayilmaz
akhilnev
abhinavsanyal
srujan-zenshastra
sundai-club
gokborayilmaz
ArmandKarimi
aparusel
debsagar
saintzema
Janata-Wifi
rohit3644
matthew1809
praveenvv
assistant-ui
igorferreira
sergioalberto
JinTanba
GriffinCanCode
jackjohns19
jaslr
joshuaheller
haashu0412
k0d3d
AbhiramChikatla
Manojkumar063
dustland
tomjuggler
thehsansaeed
EddyGiusepe
vschs007
CogitoNTNU
Naunau75
sethupavan12
cuongpham2107
Azuma413
akiblasco
matheus896
alimdsaif3
btjones-me
DineshMalem
codesandtags
moregatest
Ahmadkashif
migavel508
basuarunava
akk1to
akshaybhat2
steel-dev
shinobe179
miladnoo
kiranimmadi2
prerakmody
com2u
srankmeng
Kumrazznish
Yulikepython
SushritPasupuleti
Alex-Shanyi-Yuan
nickmitreski
TrevorBurgoyne
neagualexa
Ashwin-ER
chrisathlea
muzafferkadir
GriffinCanCode
MANI-WEBDEVE
sooraj46
Wegi
Lagozon-Technologies
Borghese-Gladiator
simranLagozon
DELTA-313
mconflitti-pbc
ajilkdev
hackingthemarkets
Alok-Kumar2005
context64ai
Vardaan-Grover
andrasat
zakinomiya
Jds2t
webhiveos
holohermet
SrinivasBathula9
R-Mohammed-Hasan
isfarbaset
vovanmozg
vkai2048
up1
datablissnet
hummusonrails
nishimotz
radioheavy
itsvetkov1
hermesthecat
kanwar19031
asimrs
Cinco29
shricastic
versantus
SahilKulkarni10
ichihara-3
guzus
hummusonrails
AlexMobiCraft
chronometer
xhiroga
HAQ-NAWAZ-MALIK
bhav09
adnanwahab
Nayeem0072
viren-vii
ruvnet
akhil-bot
amiller
AReid987
agent0ai
eagurin
niczy
Saik0s
Pavleras
HarleyCoops
startino
ris3abh
jujunjun110
ONIXION
JKOBEJ
imd8465
shamitv
saen-ai
Raushan-A
richtowndx
team-monolith-product
Michaelgathara
nakamasato
CellCS
yukiyamamuro
jaypyles
waku3awa
borkclinia
aiproductguy
browser-use
umatt1
Mike-37
hanson-cheng
asbmr69
PavelBelove
"""

# ===========================================

usernames = [u.strip() for u in usernames_input.strip().split('\n') if u.strip()]
print(f'Loaded {len(usernames)} usernames')
if GITHUB_COOKIE:
    print('GitHub cookie provided — will see profile emails')
else:
    print('No GitHub cookie — will only get emails from commit history')

Loaded 1633 usernames
No GitHub cookie — will only get emails from commit history


In [ ]:
# Run the scraper
pw, browser, page = await start_browser(github_cookie=GITHUB_COOKIE if GITHUB_COOKIE else None)
results = []
found_count = 0

try:
    for username in tqdm(usernames, desc='Scraping emails'):
        email = await scrape_email(page, username)
        results.append({'username': username, 'email': email})

        if email:
            found_count += 1
            print(f'  ✓ {username} -> {email}')
        else:
            print(f'  ✗ {username} -> not found')

        await page.wait_for_timeout(1000)  # be polite
finally:
    await stop_browser(pw, browser)

print(f'\nDone! Found {found_count}/{len(usernames)} emails')

Browser started (anonymous — profile emails will be hidden)


Scraping emails:   0%|          | 0/1633 [00:00<?, ?it/s]

  ✗ 3rdAI-admin -> not found
  ✓ ArkayaVenture -> admin@arkayaventure.co.uk
  ✗ Kaairofelipe -> not found
  ✓ cpiprint -> tcarson@cpiprint.com
  ✓ cpiprint -> tcarson@cpiprint.com
  ✓ cpiprint -> tcarson@cpiprint.com
  ✓ cpiprint -> tcarson@cpiprint.com
  ✓ shreyasgm -> shreyas.gm61@gmail.com
  ✗ asfakahamedc -> not found
  ✗ jsairdrop1 -> not found
  ✓ Kakachia777 -> your.email@example.com
  ✓ zfogg -> me@zfo.gg
  ✗ BinxAI -> not found
  ✗ aiinpocket -> not found
  ✓ GiantClam -> liulanggoukk@gmail.com
  ✓ lordwilsonDev -> wilsonlord241@gmail.com
  ✗ Mirxa27 -> not found
  ✗ jrmatherly -> not found
  ✓ anjijava16 -> anjaiahspr@gmail.com
  ✓ amp135791 -> apatel@patelcapitaladvisor.com
  ✓ danmestas -> john@example.com
  ✗ bobdavis84 -> not found
  ✓ Sealjay -> sealjay@fosstodon.org
  ✗ TeamADAPT -> not found
  ✓ David2024patton -> dpatton01@student.fullsail.edu
  ✗ yubarrdevo -> not found
  ✓ idgrou -> idgroup40@gmail.com
  ✗ mundo-labs -> not found
  ✗ brandonlacoste9-tech -> not foun

In [ ]:
# Results
df = pd.DataFrame(results)
print(f'Total: {len(df)}')
print(f'With email: {df["email"].notna().sum()}')
print(f'Without email: {df["email"].isna().sum()}')
print()

# Show all results
display(df)

# Save and download CSV
csv_filename = 'stargazer_emails.csv'
df.to_csv(csv_filename, index=False)

from google.colab import files
files.download(csv_filename)
print(f'\nDownloading {csv_filename}...')